<a href="https://colab.research.google.com/github/6pb4wnww4g-beep/Jason/blob/main/Market_ODDSAPI_CS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- JASON ODDSAPI GW1-5 IMPORT v1 ---
# Formål:
# Hente odds fra The Odds API for EPL, men KUN beholde kampene i GW1-5.
#
# Skriver til Google Sheets:
#   OddsAPI_GW1_5_Market_Check
#   OddsAPI_GW1_5_Bookmaker_Detail
#
# Etterpå kan du kjøre:
#   Jason_Market_xG_All_Fixtures_v1.py
# MEN da må INPUT_SHEET i xG-scriptet peke på:
#   OddsAPI_GW1_5_Bookmaker_Detail
#
# Viktig:
# - The Odds API returnerer bare kamper bookmakerne faktisk har priset.
# - Hvis GW2-5 ikke finnes hos OddsAPI ennå, får vi færre enn 50 kamper.
# - Scriptet skriver en tydelig status i Market_Check.

import time
import requests
import numpy as np
import pandas as pd
from google.colab import auth
import gspread
from google.auth import default

# -----------------------
# KONFIGURASJON
# -----------------------

API_KEY = "1febfae459fe024b33c82de75598bd9f"  # Lim inn nøkkelen din mellom anførselstegnene

SPREADSHEET_NAME = "Jason Development"

SPORT = "soccer_epl"
REGIONS = "uk"
MARKETS = "h2h,totals"
ODDS_FORMAT = "decimal"
DATE_FORMAT = "iso"

OUT_CHECK_SHEET = "OddsAPI_GW1_5_Market_Check"
OUT_DETAIL_SHEET = "OddsAPI_GW1_5_Bookmaker_Detail"

# Bruker navnene slik The Odds API normalt returnerer EPL-lag.
# Hvis en kamp ikke matches, se Market_Check og juster TEAM_ALIASES nederst.
GW1_5_FIXTURES = {
    1: [
        ("Arsenal", "Coventry City"),
        ("Hull City", "Manchester United"),
        ("Everton", "Crystal Palace"),
        ("Ipswich Town", "Sunderland"),
        ("Nottingham Forest", "Leeds United"),
        ("Brentford", "Tottenham Hotspur"),
        ("Manchester City", "Bournemouth"),
        ("Brighton and Hove Albion", "Aston Villa"),
        ("Newcastle United", "Liverpool"),
        ("Fulham", "Chelsea"),
    ],
    2: [
        ("Chelsea", "Brighton and Hove Albion"),
        ("Liverpool", "Nottingham Forest"),
        ("Tottenham Hotspur", "Newcastle United"),
        ("Manchester United", "Ipswich Town"),
        ("Leeds United", "Brentford"),
        ("Coventry City", "Hull City"),
        ("Sunderland", "Fulham"),
        ("Crystal Palace", "Manchester City"),
        ("Bournemouth", "Everton"),
        ("Aston Villa", "Arsenal"),
    ],
    3: [
        ("Hull City", "Aston Villa"),
        ("Arsenal", "Chelsea"),
        ("Everton", "Manchester United"),
        ("Brentford", "Sunderland"),
        ("Nottingham Forest", "Tottenham Hotspur"),
        ("Ipswich Town", "Liverpool"),
        ("Newcastle United", "Bournemouth"),
        ("Manchester City", "Coventry City"),
        ("Fulham", "Crystal Palace"),
        ("Brighton and Hove Albion", "Leeds United"),
    ],
    4: [
        ("Manchester United", "Manchester City"),
        ("Liverpool", "Fulham"),
        ("Sunderland", "Arsenal"),
        ("Leeds United", "Newcastle United"),
        ("Crystal Palace", "Ipswich Town"),
        ("Bournemouth", "Brentford"),
        ("Tottenham Hotspur", "Everton"),
        ("Coventry City", "Brighton and Hove Albion"),
        ("Aston Villa", "Nottingham Forest"),
        ("Chelsea", "Hull City"),
    ],
    5: [
        ("Aston Villa", "Liverpool"),
        ("Coventry City", "Brentford"),
        ("Crystal Palace", "Bournemouth"),
        ("Everton", "Manchester City"),
        ("Fulham", "Arsenal"),
        ("Hull City", "Leeds United"),
        ("Ipswich Town", "Chelsea"),
        ("Manchester United", "Sunderland"),
        ("Newcastle United", "Nottingham Forest"),
        ("Tottenham Hotspur", "Brighton and Hove Albion"),
    ],
}

TEAM_ALIASES = {
    # short/common -> OddsAPI likely official
    "Man City": "Manchester City",
    "Man United": "Manchester United",
    "Man Utd": "Manchester United",
    "Spurs": "Tottenham Hotspur",
    "Tottenham": "Tottenham Hotspur",
    "Brighton": "Brighton and Hove Albion",
    "Newcastle": "Newcastle United",
    "Nottingham Forest": "Nottingham Forest",
    "Nott'm Forest": "Nottingham Forest",
    "Nottm Forest": "Nottingham Forest",
    "Ipswich": "Ipswich Town",
    "Leeds": "Leeds United",
    "Coventry": "Coventry City",
    "Hull": "Hull City",
}


# -----------------------
# Google Sheets
# -----------------------

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open(SPREADSHEET_NAME)


def update_worksheet(sheet_name, dataframe):
    print(f"Oppdaterer {sheet_name}...")
    df_clean = dataframe.replace([np.inf, -np.inf], np.nan).fillna("")
    try:
        ws = sh.worksheet(sheet_name)
        ws.clear()
    except Exception:
        ws = sh.add_worksheet(title=sheet_name, rows="5000", cols="80")

    ws.resize(
        rows=max(200, len(df_clean) + 20),
        cols=max(30, len(df_clean.columns) + 5)
    )
    ws.update([df_clean.columns.tolist()] + df_clean.values.tolist())
    time.sleep(2)


# -----------------------
# Helpers
# -----------------------

def norm_team(name):
    s = str(name).strip()
    return TEAM_ALIASES.get(s, s)


def fixture_key(home, away):
    return (norm_team(home), norm_team(away))


FIXTURE_TO_GW = {}
for gw, matches in GW1_5_FIXTURES.items():
    for home, away in matches:
        FIXTURE_TO_GW[fixture_key(home, away)] = gw


def get_market_outcomes(bookmaker, market):
    outcomes = []
    for outcome in market.get("outcomes", []):
        outcomes.append({
            "bookmaker_key": bookmaker.get("key", ""),
            "bookmaker": bookmaker.get("title", ""),
            "bookmaker_last_update": bookmaker.get("last_update", ""),
            "market": market.get("key", ""),
            "market_last_update": market.get("last_update", ""),
            "outcome_name": outcome.get("name", ""),
            "price": outcome.get("price", ""),
            "point": outcome.get("point", ""),
            "description": outcome.get("description", ""),
        })
    return outcomes


# -----------------------
# Fetch odds
# -----------------------

if not API_KEY.strip():
    raise ValueError('API_KEY mangler. Lim inn nøkkelen din i API_KEY = ""')

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds"
params = {
    "apiKey": API_KEY,
    "regions": REGIONS,
    "markets": MARKETS,
    "oddsFormat": ODDS_FORMAT,
    "dateFormat": DATE_FORMAT,
}

print("Henter odds fra The Odds API...")
resp = requests.get(url, params=params, timeout=30)

print("Status code:", resp.status_code)
print("Requests remaining:", resp.headers.get("x-requests-remaining"))
print("Requests used:", resp.headers.get("x-requests-used"))

if resp.status_code != 200:
    raise RuntimeError(f"OddsAPI-feil {resp.status_code}: {resp.text[:500]}")

events = resp.json()
print(f"Antall events returnert fra OddsAPI: {len(events)}")


# -----------------------
# Filter GW1-5 and flatten
# -----------------------

check_rows = []
detail_rows = []

seen_fixture_keys = set()

for event in events:
    home = norm_team(event.get("home_team", ""))
    away = norm_team(event.get("away_team", ""))
    key = (home, away)

    gw = FIXTURE_TO_GW.get(key)

    include = gw is not None

    if include:
        seen_fixture_keys.add(key)

    bookmaker_count = len(event.get("bookmakers", []))
    market_names = sorted({
        m.get("key", "")
        for b in event.get("bookmakers", [])
        for m in b.get("markets", [])
    })

    check_rows.append({
        "include_gw1_5": include,
        "GW": gw if gw else "",
        "match_id": event.get("id", ""),
        "sport_key": event.get("sport_key", ""),
        "sport_title": event.get("sport_title", ""),
        "commence_time": event.get("commence_time", ""),
        "home_team": home,
        "away_team": away,
        "bookmakers_count": bookmaker_count,
        "markets_available": ", ".join(market_names),
        "status": "IN_GW1_5" if include else "NOT_IN_GW1_5_FIXTURE_LIST",
    })

    if not include:
        continue

    for bookmaker in event.get("bookmakers", []):
        for market in bookmaker.get("markets", []):
            for row in get_market_outcomes(bookmaker, market):
                detail_rows.append({
                    "GW": gw,
                    "match_id": event.get("id", ""),
                    "sport_key": event.get("sport_key", ""),
                    "sport_title": event.get("sport_title", ""),
                    "commence_time": event.get("commence_time", ""),
                    "home_team": home,
                    "away_team": away,
                    **row
                })


# Add missing fixtures to check
for key, gw in FIXTURE_TO_GW.items():
    if key not in seen_fixture_keys:
        home, away = key
        check_rows.append({
            "include_gw1_5": False,
            "GW": gw,
            "match_id": "",
            "sport_key": SPORT,
            "sport_title": "",
            "commence_time": "",
            "home_team": home,
            "away_team": away,
            "bookmakers_count": 0,
            "markets_available": "",
            "status": "MISSING_FROM_ODDSAPI_RESPONSE",
        })

check_df = pd.DataFrame(check_rows)
detail_df = pd.DataFrame(detail_rows)

# Sort
if not check_df.empty:
    check_df = check_df.sort_values(["GW", "commence_time", "home_team"], na_position="last").reset_index(drop=True)

if not detail_df.empty:
    detail_df = detail_df.sort_values(["GW", "commence_time", "home_team", "bookmaker", "market", "outcome_name"]).reset_index(drop=True)

update_worksheet(OUT_CHECK_SHEET, check_df)
update_worksheet(OUT_DETAIL_SHEET, detail_df)

print("Ferdig.")
print(f"Sjekk arket: {OUT_CHECK_SHEET}")
print(f"Detaljer: {OUT_DETAIL_SHEET}")
print("")
print(f"GW1-5 kamper funnet: {len(seen_fixture_keys)} av 50")
print(f"Detaljrader skrevet: {len(detail_df)}")
print("")
print("Hvis færre enn 50 kamper finnes, er resten trolig ikke priset hos OddsAPI/bookmakerne ennå.")

Henter odds fra The Odds API...
Status code: 200
Requests remaining: 494
Requests used: 6
Antall events returnert fra OddsAPI: 10
Oppdaterer OddsAPI_GW1_5_Market_Check...
Oppdaterer OddsAPI_GW1_5_Bookmaker_Detail...
Ferdig.
Sjekk arket: OddsAPI_GW1_5_Market_Check
Detaljer: OddsAPI_GW1_5_Bookmaker_Detail

GW1-5 kamper funnet: 10 av 50
Detaljrader skrevet: 627

Hvis færre enn 50 kamper finnes, er resten trolig ikke priset hos OddsAPI/bookmakerne ennå.
